# Notebook-legal.com.br

[Dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [1]:
%pip install pandas duckdb pyarrow


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import duckdb
from time import perf_counter
import datetime

In [3]:
arquivo = "yellow_tripdata_2026-01.parquet"
df = pd.read_parquet(arquivo)
con = duckdb.connect()
con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet ('{ arquivo }')
""")

In [4]:
print(f"Número de Linhas no df: {len(df)}")

con.sql("SELECT COUNT(*) FROM corridas").show()

print(f"Colunas: {df.columns}")

con.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'corridas'
""").show()

Número de Linhas no df: 3724889
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3724889 │
└──────────────┘

Colunas: Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')
┌───────────────────────┐
│      column_name      │
│        varchar        │
├───────────────────────┤
│ VendorID              │
│ tpep_pickup_datetime  │
│ tpep_dropoff_datetime │
│ passenger_count       │
│ trip_distance         │
│ RatecodeID            │
│ store_and_fwd_flag    │
│ PULocationID          │
│ DOLocationID          │
│ payment_type          │
│ fare_amount           │
│ extra                 │
│ mta_tax               │
│ ti

In [5]:
df = df[(df['tpep_pickup_datetime'] >= '2026-01-01') 
      & (df['tpep_dropoff_datetime'] < '2026-02-01')]

con.sql("""
    DELETE FROM corridas
    WHERE tpep_pickup_datetime < '2026-01-01'
    OR tpep_dropoff_datetime >= '2026-02-01';
""")

In [6]:
print(f"Número de Linhas no df: {len(df)}")
con.sql("SELECT COUNT(*) FROM corridas").show()

Número de Linhas no df: 3722601
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3722601 │
└──────────────┘



# 1) Corridas por horário (sem IA)
Para cada hora de início da corrida (0 a 23, a partir de tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance) e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o menor.

In [ ]:
tempo_duckdb = perf_counter()
results = con.sql("""
        SELECT date_part('hour', tpep_pickup_datetime) AS hour_pickup, COUNT(*) AS n_corridas, MEAN(trip_distance), MEAN(total_amount)
        FROM corridas
        GROUP BY hour_pickup
        ORDER BY n_corridas DESC
        """).fetchall()
tempo_duckdb = perf_counter() - tempo_duckdb

print(f'DUCKDB -> {tempo_duckdb:.5f}s')
print('(hour, #, mean_distance, mean_price)')
for result in results:
        print(result)

df_ = df.copy()
tempo_pandas = perf_counter()
df_['hour'] = df_['tpep_pickup_datetime'].dt.hour
pandas_results = (
        df_.groupby("hour", as_index=False)
           .agg(
                   n_corridas=('hour', 'size'),
                   mean_distance=('trip_distance', 'mean'),
                   mean_price=('total_amount', 'mean')
           )
           .sort_values('n_corridas', ascending=False)
)
tempo_pandas = perf_counter() - tempo_pandas
print(f'\n\nPANDAS -> {tempo_pandas:.5f}s')
print(pandas_results)

print(f'\n\n{('DuckDB' if tempo_duckdb < tempo_pandas else 'Pandas')} foi mais rápido {(tempo_pandas/tempo_duckdb) if tempo_duckdb < tempo_pandas else (tempo_duckdb/tempo_pandas):.2f}x')

DUCKDB -> 0.02945s
(hour, #, mean_distance, mean_price)
(18, 265574, 3.1910994299140447, 29.317995323335083)
(17, 252172, 6.149088320670003, 30.089567834650577)
(19, 234541, 4.336297193241236, 29.063148106298655)
(16, 227892, 4.9367314341881094, 30.851378591611848)
(15, 221677, 5.255240597806729, 29.340656631042748)
(20, 212883, 5.726825063532578, 29.116928829448312)
(21, 212544, 7.311580425700096, 29.230307653942127)
(14, 207506, 6.578418744518232, 29.07168284290428)
(13, 195708, 5.5268278251272385, 27.888121384919025)
(22, 194797, 5.440825269382999, 29.51084939706204)
(12, 187434, 5.295306774651327, 27.49780498735428)
(11, 171726, 7.003862781407587, 27.032680083387717)
(10, 161418, 9.127193311774397, 27.314342452513692)
(9, 156894, 7.422043545323574, 27.88476436319965)
(23, 150354, 4.015366801016266, 29.99683606688038)
(8, 148280, 8.680417655786345, 29.741728891285877)
(7, 113463, 17.34665996844786, 30.959745467685313)
(0, 111866, 4.750563441975219, 29.5634798777106)
(1, 77818, 8.245

# 4) Consulta livre (com IA)
Utilize IA para propor uma pergunta analítica diferente das anteriores e auxiliar na implementação em Pandas e SQL. Informe a pergunta escolhida e registre os principais prompts utilizados.

# 5) Corridas simultâneas (sem IA).
Considere as corridas iniciadas no dia 15/01/2026 a partir das 8h. Conte quantos pares de corridas tiveram seus períodos de viagem sobrepostos, considerando os horários de início (tpep_pickup_datetime) e de término (tpep_dropoff_datetime). Considere apenas pares de corridas distintas e conte cada par uma única vez. Execute a consulta para três janelas de tempo, das 8h às 9h, das 8h às 10h e das 8h às 12h (sem incluir o horário final). Informe a quantidade de pares e o tempo de execução de cada janela. Se
alguma execução exceder a memória disponível, registre isso e não aumente a janela.

In [ ]:
inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 09:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–09h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 10:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–10h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 12:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–12h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")




08h–09h | Corridas: 6603 | Pares sobrepostos: 9907081 | Tempo: 0.29208s
08h–10h | Corridas: 13256 | Pares sobrepostos: 23145124 | Tempo: 0.36176s
08h–12h | Corridas: 25441 | Pares sobrepostos: 46129301 | Tempo: 0.90754s
